In [39]:
# 0. imports and paths

import os
import gzip
import tarfile
import io
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import zscore, spearmanr, kruskal, mannwhitneyu
from scipy.optimize import nnls
import warnings
warnings.filterwarnings('ignore')

base        = 'D:/TNBC_SV_DNA_Repair'
data_dir    = os.path.join(base, 'dataset')
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

group_order  = ['Low', 'Moderate', 'High']
group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}

print('paths set')
print('panel genes:', len(all_panel))

paths set
panel genes: 23


In [40]:
# 1. load all prior results

gips_df    = pd.read_csv(os.path.join(scores_dir, 'gips_scores.csv'))
gene_score = pd.read_csv(os.path.join(scores_dir, 'gene_disruption_scores.csv'), index_col=0)
expr_tnbc  = pd.read_csv(os.path.join(tables_dir, 'expr_tnbc.csv'), index_col=0)
cn_tnbc    = pd.read_csv(os.path.join(tables_dir, 'cn_tnbc.csv'), index_col=0)
surv_tnbc  = pd.read_csv(os.path.join(tables_dir, 'surv_tnbc.csv'))
mut_tnbc   = pd.read_csv(os.path.join(tables_dir, 'mut_tnbc.csv'))
immune_gips = pd.read_csv(os.path.join(tables_dir, 'immune_gips_merged.csv'))
gene_expr_panel = pd.read_csv(os.path.join(tables_dir, 'gene_expression_panel.csv'), index_col=0)

tnbc_samples = gips_df['sample'].tolist()

print('gips_df:', gips_df.shape)
print('gene_score:', gene_score.shape)
print('expr_tnbc:', expr_tnbc.shape)
print('tnbc samples:', len(tnbc_samples))
print('GIPS groups:', gips_df['GIPS_group'].value_counts().to_dict())

gips_df: (121, 8)
gene_score: (23, 121)
expr_tnbc: (26, 121)
tnbc samples: 121
GIPS groups: {'Low': 52, 'High': 36, 'Moderate': 33}


In [41]:
# 2. Section 1: TNBC Classification Validation
# Note: TCGA-BRCA clinical phenotype file (BRCA_clinicalMatrix) not in dataset.
# Validation uses expression-based receptor status across all 1218 TCGA-BRCA samples.
# gene_expression_panel.csv contains ESR1, PGR, ERBB2 expression for all samples.

# Load full-cohort receptor expression (all 1218 samples)
esr1  = gene_expr_panel.loc['ESR1']    # log2 expression
pgr   = gene_expr_panel.loc['PGR']
erbb2 = gene_expr_panel.loc['ERBB2']

# Original TNBC definition: below-median for all three (as in notebook 1)
esr1_thresh  = esr1.median()
pgr_thresh   = pgr.median()
erbb2_thresh = erbb2.median()

orig_tnbc_mask = ((esr1 < esr1_thresh) & (pgr < pgr_thresh) & (erbb2 < erbb2_thresh))
orig_tnbc_set  = set(esr1.index[orig_tnbc_mask].tolist())

print(f'ESR1 threshold: {round(esr1_thresh,3)}')
print(f'PGR threshold: {round(pgr_thresh,3)}')
print(f'ERBB2 threshold: {round(erbb2_thresh,3)}')
print(f'Samples below all thresholds: {len(orig_tnbc_set)}')
print(f'Our 121 TNBC cohort subset of threshold-pass: {len(set(tnbc_samples) & orig_tnbc_set)}')

ESR1 threshold: 1.882
PGR threshold: 0.156
ERBB2 threshold: 0.028
Samples below all thresholds: 132
Our 121 TNBC cohort subset of threshold-pass: 121


In [42]:
# 3. TNBC validation: z-score-based re-classification
# Compare median-threshold calls vs. z-score < -0.5 calls as independent method

# z-score approach (population-level z-score across all 1218 samples)
esr1_z  = (esr1  - esr1.mean())  / esr1.std()
pgr_z   = (pgr   - pgr.mean())   / pgr.std()
erbb2_z = (erbb2 - erbb2.mean()) / erbb2.std()

# z-score TNBC: all three z < 0 (below mean)
zscore_tnbc_mask = ((esr1_z < 0) & (pgr_z < 0) & (erbb2_z < 0))
zscore_tnbc_set  = set(esr1.index[zscore_tnbc_mask].tolist())

# Focus on our 121-sample cohort
cohort_set  = set(tnbc_samples)
all_samples = set(esr1.index.tolist())
non_tnbc    = all_samples - orig_tnbc_set

# 2x2 confusion matrix: rows = our classification, cols = z-score validation
# True Positives: in our cohort AND z-score confirms TNBC
TP = len(cohort_set & zscore_tnbc_set)
FP = len(cohort_set - zscore_tnbc_set)   # our TNBC, z-score says non-TNBC
FN = len((orig_tnbc_set - cohort_set) & zscore_tnbc_set)  # missed TNBC
TN = len(non_tnbc - zscore_tnbc_set)

concordance = TP / len(cohort_set) if len(cohort_set) > 0 else 0

print('2x2 confusion matrix (our calls vs z-score validation):')
print(f'  True Positives (TNBC confirmed): {TP}')
print(f'  False Positives (TNBC, z-score non-TNBC): {FP}')
print(f'  True Negatives: {TN}')
print(f'  False Negatives: {FN}')
print(f'Concordance within our 121-sample cohort: {round(concordance*100, 1)}%')

if concordance > 0.85:
    print('Concordance >85%: expression-threshold classification confirmed')
else:
    print('Concordance <85%: reporting both full and confirmed-subset analyses')

2x2 confusion matrix (our calls vs z-score validation):
  True Positives (TNBC confirmed): 107
  False Positives (TNBC, z-score non-TNBC): 14
  True Negatives: 959
  False Negatives: 8
Concordance within our 121-sample cohort: 88.4%
Concordance >85%: expression-threshold classification confirmed


In [43]:
# 4. receptor expression distribution plot for 121 TNBC samples

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, gene, expr_all, thresh in [
    (axes[0], 'ESR1',  esr1,  esr1_thresh),
    (axes[1], 'PGR',   pgr,   pgr_thresh),
    (axes[2], 'ERBB2', erbb2, erbb2_thresh)
]:
    tnbc_vals    = expr_all[tnbc_samples].values
    nontnbc_vals = expr_all[[s for s in expr_all.index if s not in tnbc_samples]].values
    ax.hist(nontnbc_vals, bins=30, alpha=0.5, color='steelblue', label='non-TNBC', density=True)
    ax.hist(tnbc_vals,    bins=20, alpha=0.7, color='salmon',    label='TNBC (n=121)', density=True)
    ax.axvline(thresh, color='black', linestyle='--', linewidth=1.2, label=f'threshold={round(thresh,2)}')
    ax.set_title(f'{gene} expression')
    ax.set_xlabel('log2 expression')
    ax.legend(fontsize=7)

plt.suptitle('Receptor expression: TNBC vs non-TNBC', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb5_receptor_validation.png'), dpi=150)
plt.show()
print('saved receptor validation plot')

saved receptor validation plot


In [44]:
# 5. save classification validation results

confusion = pd.DataFrame({
    'metric': ['TP_tnbc_confirmed', 'FP_tnbc_not_confirmed', 'TN', 'FN_missed_tnbc',
               'concordance_pct', 'cohort_size', 'zscore_tnbc_total'],
    'value': [TP, FP, TN, FN, round(concordance*100, 2), len(cohort_set), len(zscore_tnbc_set)]
})
confusion.to_csv(os.path.join(tables_dir, 'tnbc_classification_validation.csv'), index=False)
print('Saved tnbc_classification_validation.csv')
print(confusion)

Saved tnbc_classification_validation.csv
                  metric   value
0      TP_tnbc_confirmed  107.00
1  FP_tnbc_not_confirmed   14.00
2                     TN  959.00
3         FN_missed_tnbc    8.00
4        concordance_pct   88.43
5            cohort_size  121.00
6      zscore_tnbc_total  242.00


In [55]:
# 6. Section 2: Methylation - position-based probe discovery
# Scan file and collect all probe IDs, then match to panel genes via mygene.info coordinates

import urllib.request
import json

meth_file = os.path.join(data_dir, 'TCGA.BRCA.sampleMap_HumanMethylation450.gz')
print(f'Methylation file exists: {os.path.exists(meth_file)}')
print(f'File size: {round(os.path.getsize(meth_file)/1e9, 2)} GB')

# Gene coordinates from notebook 1 mygene.info lookup - GRCh38
GENE_COORDS = {
    'BRCA1':   ('17', 43044295, 43125364),
    'BRCA2':   ('13', 32315086, 32400268),
    'PALB2':   ('16', 23603160, 23641310),
    'RAD51':   ('15', 40695556, 40732382),
    'RAD51B':  ('14', 68155591, 68536655),
    'RAD51C':  ('17', 58692573, 58743680),
    'RAD51D':  ('17', 33434237, 33465598),
    'BRIP1':   ('17', 61679107, 61863422),
    'ATM':     ('11', 108222832, 108369102),
    'CHEK2':   ('22', 28687742, 28742422),
    'STAG2':   ('X',  123870913, 124036531),
    'STAG3':   ('7',  99954023, 100068890),
    'SMC1A':   ('X',  53441236, 53497834),
    'SMC1B':   ('22', 41637534, 41681546),
    'RAD21':   ('8',  117700551, 117729962),
    'REC8':    ('14', 24428449, 24477611),
    'HORMAD1': ('1',  118011746, 118066987),
    'HORMAD2': ('22', 30378023, 30443234),
    'SYCP2':   ('20', 48267955, 48437750),
    'SYCP3':   ('12', 34014574, 34068234),
    'MLH3':    ('14', 75526876, 75617052),
    'MSH4':    ('1',  109781450, 109826476),
    'MSH5':    ('6',  31883978, 31942990),
}

# Fetch Illumina 450k manifest probe coordinates from UCSC via API
# We query the UCSC hg19 track for HumanMethylation450 probe positions
# Then lift over or use hg38 positions

print('\nFetching probe coordinates from Illumina manifest via mygene.info...')
print('This will match probes to panel gene promoter windows (TSS +/- 2000bp)')

# Define promoter windows: TSS +/- 2000 bp for each gene
PROMOTER_WINDOWS = {}
for gene, (chrom, start, end) in GENE_COORDS.items():
    # Use both TSS (start) and extend upstream
    tss = start  # using start as TSS approximation
    PROMOTER_WINDOWS[gene] = (chrom, max(0, tss - 2000), tss + 500)

print('Promoter windows defined for', len(PROMOTER_WINDOWS), 'genes')
print('Example BRCA1 window:', PROMOTER_WINDOWS['BRCA1'])

Methylation file exists: True
File size: 0.78 GB

Fetching probe coordinates from Illumina manifest via mygene.info...
This will match probes to panel gene promoter windows (TSS +/- 2000bp)
Promoter windows defined for 23 genes
Example BRCA1 window: ('17', 43042295, 43044795)


In [56]:
# 7. Fetch Illumina 450k probe positions and match to panel gene promoters

# Download probe coordinate table from UCSC Xena (small file, probe annotations)
probe_annot_url = 'https://raw.githubusercontent.com/patterning/methylation-tools/master/HM450.hg38.manifest.tsv.gz'

# Alternative: use the Illumina manifest hosted on GitHub
# We use a curated hg38 probe position table
probe_annot_file = os.path.join(data_dir, 'HM450_hg38_probes.tsv.gz')

if not os.path.exists(probe_annot_file):
    print('Downloading Illumina 450k hg38 probe annotations...')
    url = 'https://zhouserver.research.chop.edu/InfiniumAnnotation/20180909/HM450/HM450.hg38.manifest.tsv.gz'
    try:
        urllib.request.urlretrieve(url, probe_annot_file)
        print(f'Downloaded: {round(os.path.getsize(probe_annot_file)/1e6,1)} MB')
    except Exception as ex:
        print(f'Download failed: {ex}')
        probe_annot_file = None
else:
    print(f'Probe annotation file exists: {round(os.path.getsize(probe_annot_file)/1e6,1)} MB')

if probe_annot_file and os.path.exists(probe_annot_file):
    with gzip.open(probe_annot_file, 'rt') as f:
        probe_annot = pd.read_csv(f, sep='\t', low_memory=False)
    print('Probe annotation shape:', probe_annot.shape)
    print('Columns (first 10):', list(probe_annot.columns[:10]))

Downloaded: 28.9 MB
Probe annotation shape: (485577, 57)
Columns (first 10): ['CpG_chrm', 'CpG_beg', 'CpG_end', 'probe_strand', 'probeID', 'address_A', 'address_B', 'channel', 'designType', 'nextBase']


In [59]:
# 8. Match probes using wider window around both gene ends

PROBE_GENE_MAP = {}
probe_targets = set()

for gene, (chrom, start, end) in GENE_COORDS.items():
    chrom_str = str(chrom)
    
    # Wide window: 5000bp upstream of both possible TSS (start and end)
    # This handles both plus and minus strand genes
    windows = [
        (max(0, start - 5000), start + 1000),  # upstream of start
        (max(0, end - 1000),   end + 5000),    # upstream of end (minus strand TSS)
    ]
    
    matched_all = set()
    for w_start, w_end in windows:
        mask = (
            (probe_annot['chrom'] == chrom_str) &
            (probe_annot['CpG_beg'] >= w_start) &
            (probe_annot['CpG_beg'] <= w_end)
        )
        matched_all.update(probe_annot[mask]['probeID'].tolist())
    
    # Cap at 10 probes per gene to avoid ATM-style overload
    matched_list = sorted(matched_all)[:10]
    PROBE_GENE_MAP[gene] = matched_list
    probe_targets.update(matched_list)
    print(f'{gene}: {len(matched_list)} probes (from {len(matched_all)} candidates)')

print(f'\nTotal unique probe targets: {len(probe_targets)}')
print(f'Genes with at least one probe: {sum(1 for v in PROBE_GENE_MAP.values() if len(v) > 0)}')
print(f'Genes with no probes: {sum(1 for v in PROBE_GENE_MAP.values() if len(v) == 0)}')
missing = [g for g, v in PROBE_GENE_MAP.items() if len(v) == 0]
print(f'Still missing: {missing}')

BRCA1: 10 probes (from 48 candidates)
BRCA2: 10 probes (from 23 candidates)
PALB2: 10 probes (from 17 candidates)
RAD51: 10 probes (from 18 candidates)
RAD51B: 1 probes (from 1 candidates)
RAD51C: 10 probes (from 16 candidates)
RAD51D: 0 probes (from 0 candidates)
BRIP1: 9 probes (from 9 candidates)
ATM: 10 probes (from 47 candidates)
CHEK2: 10 probes (from 16 candidates)
STAG2: 0 probes (from 0 candidates)
STAG3: 2 probes (from 2 candidates)
SMC1A: 7 probes (from 7 candidates)
SMC1B: 10 probes (from 11 candidates)
RAD21: 1 probes (from 1 candidates)
REC8: 3 probes (from 3 candidates)
HORMAD1: 0 probes (from 0 candidates)
HORMAD2: 2 probes (from 2 candidates)
SYCP2: 0 probes (from 0 candidates)
SYCP3: 0 probes (from 0 candidates)
MLH3: 7 probes (from 7 candidates)
MSH4: 0 probes (from 0 candidates)
MSH5: 10 probes (from 61 candidates)

Total unique probe targets: 122
Genes with at least one probe: 17
Genes with no probes: 6
Still missing: ['RAD51D', 'STAG2', 'HORMAD1', 'SYCP2', 'SYCP3'

In [60]:
# 9. Stream methylation file and extract matched probes

meth_rows  = {}
meth_cols  = None
line_count = 0
found_count = 0

with gzip.open(meth_file, 'rt') as f:
    for line in f:
        line_count += 1
        parts = line.rstrip('\n').split('\t')
        if line_count == 1:
            meth_cols = parts[1:]
            continue
        probe_id = parts[0]
        if probe_id in probe_targets:
            vals = []
            for v in parts[1:]:
                try:
                    vals.append(float(v))
                except ValueError:
                    vals.append(np.nan)
            meth_rows[probe_id] = vals
            found_count += 1
        if line_count % 100000 == 0:
            print(f'  scanned {line_count//1000}k rows, found {found_count} so far')

print(f'Total rows scanned: {line_count}')
print(f'Probes found: {found_count} / {len(probe_targets)}')
print(f'Sample columns: {len(meth_cols)}')

# Build DataFrame
meth_df   = pd.DataFrame(meth_rows, index=meth_cols).T
print('Methylation matrix shape (probes x samples):', meth_df.shape)

# Subset to TNBC samples
tnbc_in_meth = [s for s in tnbc_samples if s in meth_df.columns]
meth_tnbc    = meth_df[tnbc_in_meth]
print('TNBC samples with methylation:', len(tnbc_in_meth))

  scanned 100k rows, found 21 so far
  scanned 200k rows, found 53 so far
  scanned 300k rows, found 74 so far
  scanned 400k rows, found 102 so far
Total rows scanned: 485578
Probes found: 122 / 122
Sample columns: 888
Methylation matrix shape (probes x samples): (122, 888)
TNBC samples with methylation: 86


In [ ]:
# compare sample ID formats

print('Methylation sample IDs (first 3):', meth_tnbc.columns[:3].tolist())
print('gene_score sample IDs (first 3):', gene_score.columns[:3].tolist())
print('tnbc_samples (first 3):', tnbc_samples[:3])

# Check overlap
meth_cols_set = set(meth_tnbc.columns)
gene_score_cols = set(gene_score.columns)
tnbc_set = set(tnbc_samples)

print(f'\nmeth_tnbc columns: {len(meth_cols_set)}')
print(f'gene_score columns: {len(gene_score_cols)}')
print(f'tnbc_samples: {len(tnbc_set)}')
print(f'meth vs tnbc_samples overlap: {len(meth_cols_set & tnbc_set)}')
print(f'meth vs gene_score overlap: {len(meth_cols_set & gene_score_cols)}')

Methylation sample IDs (first 3): ['TCGA-A1-A0SK-01', 'TCGA-A1-A0SM-01', 'TCGA-A1-A0SO-01']
gene_score sample IDs (first 3): ['TCGA-A1-A0SK-01', 'TCGA-A1-A0SM-01', 'TCGA-A1-A0SO-01']
tnbc_samples (first 3): ['TCGA-A1-A0SK-01', 'TCGA-A1-A0SM-01', 'TCGA-A1-A0SO-01']

meth_tnbc columns: 86
gene_score columns: 121
tnbc_samples: 121
meth vs tnbc_samples overlap: 86
meth vs gene_score overlap: 86


In [64]:
# 10. compute per-gene mean beta and methylation disruption indicator

gene_beta = {}
for gene, probes in PROBE_GENE_MAP.items():
    found = [p for p in probes if p in meth_tnbc.index]
    if len(found) == 0:
        gene_beta[gene] = pd.Series(np.nan, index=meth_tnbc.columns)
    else:
        gene_beta[gene] = meth_tnbc.loc[found].mean(axis=0)

gene_beta_df = pd.DataFrame(gene_beta).T  # genes x 86 samples
print('gene_beta_df shape:', gene_beta_df.shape)
print('Genes with data:', gene_beta_df.notna().any(axis=1).sum())

# z-score per gene across 86 TNBC samples with methylation
def zscore_row(row):
    vals = row.values.astype(float)
    m = np.nanmean(vals)
    s = np.nanstd(vals)
    if s == 0 or np.isnan(s):
        return pd.Series(0.0, index=row.index)
    return pd.Series((vals - m) / s, index=row.index)

meth_z    = gene_beta_df.apply(zscore_row, axis=1)
meth_disr = (meth_z.abs() > 1.96).astype(float)
meth_disr = meth_disr.fillna(0)

# Only use the 86 samples that have methylation data
common_samps_meth = meth_tnbc.columns.tolist()
print('TNBC samples with methylation disruption scores:', len(common_samps_meth))
print('Methylation disruption shape:', meth_disr.shape)
print('Mean disruption rate per gene (top 10):')
print(meth_disr.mean(axis=1).sort_values(ascending=False).head(10))

gene_beta_df shape: (23, 86)
Genes with data: 17
TNBC samples with methylation disruption scores: 86
Methylation disruption shape: (23, 86)
Mean disruption rate per gene (top 10):
MLH3       0.081395
RAD21      0.081395
ATM        0.081395
SMC1B      0.069767
BRCA1      0.069767
BRCA2      0.058140
RAD51      0.046512
HORMAD2    0.046512
BRIP1      0.046512
RAD51B     0.034884
dtype: float64


In [65]:
# 11. update gene-level disruption score to average four indicators
# Existing: gene_score = avg(expression, CN, mutation) per gene per sample
# New: avg(expression, CN, mutation, methylation)

# Load component scores from notebook 2
expr_disrupted = (pd.read_csv(os.path.join(tables_dir, 'expr_tnbc.csv'), index_col=0)
                  .loc[all_panel, gene_score.columns]
                  .apply(lambda row: ((row - row.mean())/row.std()).abs() > 1.96, axis=1)
                  .astype(float))

cn_disrupted = (pd.read_csv(os.path.join(tables_dir, 'cn_tnbc.csv'), index_col=0)
                .loc[all_panel, gene_score.columns] != 0).astype(float)

mut_tnbc_raw = pd.read_csv(os.path.join(tables_dir, 'mut_tnbc.csv'))
mut_mat = pd.DataFrame(0, index=all_panel, columns=gene_score.columns)
for _, row in mut_tnbc_raw.iterrows():
    if row['gene'] in all_panel and row['sample'] in mut_mat.columns:
        mut_mat.loc[row['gene'], row['sample']] = 1.0

# Align methylation to same genes x samples
meth_4  = meth_disr.reindex(index=all_panel, columns=gene_score.columns).fillna(0)

# 4-modality gene disruption score
gene_score_4 = (expr_disrupted + cn_disrupted + mut_mat + meth_4) / 4.0

print('4-modality gene disruption score shape:', gene_score_4.shape)
print('Score range:', round(gene_score_4.values.min(),3), 'to', round(gene_score_4.values.max(),3))

4-modality gene disruption score shape: (23, 121)
Score range: 0.0 to 0.75


In [66]:
# 12. recompute summed GIPS and apply min-max rescaling

gips4_raw    = gene_score_4.sum(axis=0)
gips4_min    = gips4_raw.min()
gips4_max    = gips4_raw.max()
gips4_scaled = (gips4_raw - gips4_min) / (gips4_max - gips4_min)

t33_4 = gips4_scaled.quantile(0.333)
t66_4 = gips4_scaled.quantile(0.667)

def assign4(x):
    if x <= t33_4: return 'Low'
    elif x <= t66_4: return 'Moderate'
    return 'High'

gips4_df = pd.DataFrame({
    'sample':       gene_score_4.columns,
    'GIPS4_raw':    gips4_raw.values,
    'GIPS4_scaled': gips4_scaled.values,
    'GIPS4_group':  gips4_scaled.apply(assign4).values,
})

print('4-modality GIPS groups:', gips4_df['GIPS4_group'].value_counts().to_dict())
print('Concordance with 3-modality groups:')
conc = gips4_df.merge(gips_df[['sample','GIPS_group']], on='sample')
print(pd.crosstab(conc['GIPS_group'], conc['GIPS4_group']))

4-modality GIPS groups: {'Low': 46, 'Moderate': 39, 'High': 36}
Concordance with 3-modality groups:
GIPS4_group  High  Low  Moderate
GIPS_group                      
High           34    0         2
Low             1   46         5
Moderate        1    0        32


In [67]:
# 13. disruption heatmap sorted by 4-modality GIPS

sorted_samples = gips4_df.sort_values('GIPS4_scaled')['sample'].tolist()
sorted_samples = [s for s in sorted_samples if s in gene_score_4.columns]

plot_mat = gene_score_4[sorted_samples]

# color bar annotations for GIPS group
group_bar = gips4_df.set_index('sample').loc[sorted_samples, 'GIPS4_group']
bar_colors = [group_colors[g] for g in group_bar]

fig = plt.figure(figsize=(16, 7))
gs  = plt.GridSpec(2, 1, height_ratios=[0.04, 1], hspace=0.02)
ax_bar = fig.add_subplot(gs[0])
ax_heat = fig.add_subplot(gs[1])

ax_bar.bar(range(len(sorted_samples)), [1]*len(sorted_samples), color=bar_colors, width=1)
ax_bar.set_xlim(0, len(sorted_samples))
ax_bar.axis('off')

sns.heatmap(
    plot_mat,
    cmap='RdYlBu_r',
    vmin=0, vmax=1,
    yticklabels=True,
    xticklabels=False,
    linewidths=0,
    ax=ax_heat,
    cbar_kws={'label': 'disruption score (4-modality)'}
)
ax_heat.set_xlabel(f'patients (n={len(sorted_samples)}, sorted by GIPS)')
ax_heat.set_title('4-Modality gene disruption heatmap (expression + CN + mutation + methylation)')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb5_heatmap_4modality.png'), dpi=150)
plt.show()
print('saved 4-modality heatmap')

saved 4-modality heatmap


In [68]:
# 14. pathway enrichment for 4-modality GIPS

import gseapy as gp

high4 = gips4_df[gips4_df['GIPS4_group']=='High']['sample'].tolist()
low4  = gips4_df[gips4_df['GIPS4_group']=='Low']['sample'].tolist()
high4 = [s for s in high4 if s in gene_score_4.columns]
low4  = [s for s in low4  if s in gene_score_4.columns]

diff4 = gene_score_4[high4].mean(axis=1) - gene_score_4[low4].mean(axis=1)
diff4 = diff4.sort_values(ascending=False)
top4_genes = diff4[diff4 > 0].index.tolist()

print('Genes most disrupted in high-GIPS (4-modality):')
print(diff4.head(10))

enrich4 = {}
for lib in ['GO_Biological_Process_2023', 'KEGG_2021_Human']:
    try:
        enr = gp.enrichr(gene_list=top4_genes, gene_sets=lib, outdir=None, no_plot=True)
        res = enr.results.sort_values('Adjusted P-value')
        enrich4[lib] = res
        print(f'{lib} top hits:')
        print(res[['Term','Overlap','Adjusted P-value']].head(5).to_string())
    except Exception as ex:
        print(f'{lib}: error {ex}')

Genes most disrupted in high-GIPS (4-modality):
MLH3      0.263587
RAD51B    0.216486
REC8      0.214976
RAD51     0.200181
MSH5      0.192633
SYCP3     0.178744
BRIP1     0.172101
STAG3     0.161534
PALB2     0.157005
SMC1B     0.148249
dtype: float64
GO_Biological_Process_2023 top hits:
                                                                   Term Overlap  Adjusted P-value
0                               Double-Strand Break Repair (GO:0006302)  11/168      2.834783e-15
1                                               DNA Repair (GO:0006281)  12/291      9.123666e-15
2                        Meiotic Sister Chromatid Cohesion (GO:0051177)    6/10      1.720263e-14
3  Double-Strand Break Repair Via Homologous Recombination (GO:0000724)   8/111      1.728419e-11
4                                        DNA Recombination (GO:0006310)    6/42      2.519584e-10
KEGG_2021_Human: error Error sending gene list, status code: 429


In [69]:
# 15. compare pathway enrichment 3-modality vs 4-modality

old_go = pd.read_csv(os.path.join(tables_dir, 'enrichment_GO_Biological_Process_2023.csv'))
old_kegg = pd.read_csv(os.path.join(tables_dir, 'enrichment_KEGG_2021_Human.csv'))

print('=== 3-modality top GO terms ===')
print(old_go[['Term','Adjusted P-value']].head(5).to_string())

if 'GO_Biological_Process_2023' in enrich4:
    new_go = enrich4['GO_Biological_Process_2023']
    print('\n=== 4-modality top GO terms ===')
    print(new_go[['Term','Adjusted P-value']].head(5).to_string())

    # genes in top terms
    old_top = set(old_go.head(5)['Term'])
    new_top = set(new_go.head(5)['Term'])
    print(f'\nOverlapping top terms: {len(old_top & new_top)} / 5')

=== 3-modality top GO terms ===
                                                                   Term  Adjusted P-value
0                               Double-Strand Break Repair (GO:0006302)      2.834783e-15
1                                               DNA Repair (GO:0006281)      9.123666e-15
2                        Meiotic Sister Chromatid Cohesion (GO:0051177)      1.720263e-14
3  Double-Strand Break Repair Via Homologous Recombination (GO:0000724)      1.728419e-11
4                                        DNA Recombination (GO:0006310)      2.519584e-10

=== 4-modality top GO terms ===
                                                                   Term  Adjusted P-value
0                               Double-Strand Break Repair (GO:0006302)      2.834783e-15
1                                               DNA Repair (GO:0006281)      9.123666e-15
2                        Meiotic Sister Chromatid Cohesion (GO:0051177)      1.720263e-14
3  Double-Strand Break Repair Via H

In [70]:
# 16. save 4-modality results

gips4_df.to_csv(os.path.join(scores_dir, 'gips_scores_4modality.csv'), index=False)
gene_score_4.to_csv(os.path.join(scores_dir, 'gene_disruption_scores_4modality.csv'))

for lib, df in enrich4.items():
    df.to_csv(os.path.join(tables_dir, f'enrichment_4mod_{lib}.csv'), index=False)

meth_genes_found = [g for g in all_panel
                                       if g in meth_disr.index and not meth_disr.loc[g, common_samps_meth].eq(0).all()]

print('Genes with methylation data:', len(meth_genes_found))
print('4-modality GIPS saved to scores/')
print('4-modality gene disruption saved to scores/')

Genes with methylation data: 17
4-modality GIPS saved to scores/
4-modality gene disruption saved to scores/


In [71]:
# 17. Section 3: Improved Immune Deconvolution - CIBERSORT NNLS
# Extract LM22 marker gene expression from the raw TCGA BRCA file
# (expr_tnbc.csv only has 26 panel genes; NNLS requires immune markers)

# LM22 compact reference: 22 cell types, exclusive marker genes
LM22_markers = {
    'CD8_T_cells':            ['CD8A','CD8B','GZMH','PRF1'],
    'CD4_naive_T':            ['CCR7','SELL','TCF7','LEF1'],
    'CD4_memory_resting':     ['IL7R','GATA3','S100A4'],
    'CD4_memory_activated':   ['ICOS','CD44','CD69'],
    'T_follicular_helper':    ['CXCR5','BCL6','IL21'],
    'T_regulatory':           ['FOXP3','IL2RA','IKZF2','TNFRSF18'],
    'T_gamma_delta':          ['TRDC','TRGC1','KLRB1'],
    'NK_resting':             ['KIR2DL3','KIR3DL2','KIR3DL1','KLRD1'],
    'NK_activated':           ['NKG7','GNLY','GZMB'],
    'B_naive':                ['CD19','MS4A1','CD22','BANK1'],
    'B_memory':               ['CD27','CR2','IGHD'],
    'Plasma_cells':           ['IGHG1','JCHAIN','MZB1','SDC1','PRDM1'],
    'Monocytes':              ['CD14','LYZ','CSF1R','S100A8'],
    'M0_macrophages':         ['CD68','MRC1','FCGR1A'],
    'M1_macrophages':         ['CD80','IDO1','CXCL9','CXCL10'],
    'M2_macrophages':         ['CD163','CCL18','TGFB1','ARG1'],
    'DC_resting':             ['CX3CR1','CD1C','SIRPA'],
    'DC_activated':           ['LAMP3','CD40','CCL19'],
    'Mast_resting':           ['KIT','MS4A2','TPSAB1','CPA3'],
    'Mast_activated':         ['IL6','CXCL8','CCL2','PTGS2'],
    'Eosinophils':            ['PRG2','IL5RA','SIGLEC8','EPX'],
    'Neutrophils':            ['CXCR1','CXCR2','FCGR3B','ELANE'],
}

# Additional exhaustion composite genes
exhaustion_markers = ['LAG3','HAVCR2','TIGIT','PDCD1']

all_lm22_genes = list(set(
    g for gs in LM22_markers.values() for g in gs
) | set(exhaustion_markers))

print(f'Total unique LM22 + exhaustion genes needed: {len(all_lm22_genes)}')

Total unique LM22 + exhaustion genes needed: 84


In [72]:
# 18. query gene coordinates and extract from raw expression file

import mygene
mg = mygene.MyGeneInfo()

lm22_query = mg.querymany(
    all_lm22_genes,
    scopes='symbol',
    fields='symbol,genomic_pos',
    species='human',
    as_dataframe=True
)
lm22_query = lm22_query[~lm22_query.index.duplicated(keep='first')]
lm22_query = lm22_query.dropna(subset=['genomic_pos.chr','genomic_pos.start','genomic_pos.end'])

print('Genes with coordinates:', len(lm22_query))
print('Missing coordinates:', [g for g in all_lm22_genes if g not in lm22_query.index])

3 input query terms found dup hits:	[('IGHG1', 2), ('IGHD', 2), ('TRDC', 2)]


Genes with coordinates: 77
Missing coordinates: ['KIR3DL1', 'KIR2DL3', 'KIR3DL2', 'PDCD1', 'ELANE', 'CCL18', 'TRGC1']


In [73]:
# 19. stream TCGA BRCA expression to extract LM22 + exhaustion genes for 121 TNBC samples

tcga_expr_file = os.path.join(data_dir, 'TCGA.BRCA.sampleMap_HiSeqV2_exon.gz')

# Build coordinate lookup: gene -> (chr, start, end)
gene_coords = {}
for gene in lm22_query.index:
    row = lm22_query.loc[gene]
    gene_coords[gene] = (
        str(row['genomic_pos.chr']).strip(),
        int(row['genomic_pos.start']),
        int(row['genomic_pos.end'])
    )

# Build coord -> gene reverse map for fast lookup during streaming
# (We accumulate by summing exon rows for each gene)
lm22_expr_rows = {g: None for g in gene_coords}
lm22_expr_count = {g: 0 for g in gene_coords}

tnbc_set = set(tnbc_samples)
col_order = None
tnbc_idx  = None

with gzip.open(tcga_expr_file, 'rt') as f:
    for i, line in enumerate(f):
        parts = line.rstrip('\n').split('\t')
        if i == 0:
            col_order = parts[1:]
            tnbc_idx  = [j for j, s in enumerate(col_order) if s in tnbc_set]
            col_order = [col_order[j] for j in tnbc_idx]
            continue
        coord = parts[0]
        try:
            c  = coord.split(':')[0].replace('chr','')
            se = coord.split(':')[1].split('-')
            s  = int(se[0])
            e  = int(se[1])
        except:
            continue
        for gene, (gc, gs, ge) in gene_coords.items():
            if c == gc and not (e < gs or s > ge):
                vals = [float(parts[j+1]) if j+1 < len(parts) else 0.0 for j in tnbc_idx]
                if lm22_expr_rows[gene] is None:
                    lm22_expr_rows[gene] = np.array(vals, dtype=float)
                else:
                    lm22_expr_rows[gene] += np.array(vals, dtype=float)
                lm22_expr_count[gene] += 1
        if i % 50000 == 0 and i > 0:
            found = sum(1 for v in lm22_expr_rows.values() if v is not None)
            print(f'  row {i//1000}k, genes found: {found}/{len(gene_coords)}')

# Average over exon rows per gene
lm22_expr = {}
for gene, vals in lm22_expr_rows.items():
    if vals is not None and lm22_expr_count[gene] > 0:
        lm22_expr[gene] = vals / lm22_expr_count[gene]

lm22_expr_df = pd.DataFrame(lm22_expr, index=col_order).T
lm22_expr_df = np.log2(lm22_expr_df + 1)

print('LM22 expression extracted:', lm22_expr_df.shape)
print('Genes found:', list(lm22_expr_df.index))

  row 50k, genes found: 29/77
  row 100k, genes found: 31/77
  row 150k, genes found: 39/77
  row 200k, genes found: 41/77
LM22 expression extracted: (42, 121)
Genes found: ['CD163', 'CD19', 'KLRB1', 'CD44', 'CSF1R', 'CCR7', 'CXCR5', 'LEF1', 'ARG1', 'SIGLEC8', 'CD27', 'IL2RA', 'HAVCR2', 'BANK1', 'LAG3', 'CD22', 'CD69', 'ICOS', 'TGFB1', 'IL7R', 'CD8B', 'SDC1', 'SIRPA', 'CD8A', 'GZMB', 'KLRD1', 'SELL', 'CCL2', 'TCF7', 'GATA3', 'CD80', 'IL21', 'CPA3', 'GZMH', 'IL5RA', 'FCGR1A', 'CD68', 'CCL19', 'MRC1', 'LAMP3', 'S100A8', 'MS4A1']


In [74]:
# 20. build LM22 signature matrix and run NNLS

# Genes present in both LM22 markers and our extracted expression
avail_genes = set(lm22_expr_df.index)

# Build signature matrix: rows=genes, cols=cell_types
# Value = 1.0 if gene is marker for cell type, else 0.0
cell_types = list(LM22_markers.keys())
sig_genes  = sorted(list(avail_genes & set(g for gs in LM22_markers.values() for g in gs)))

print(f'Signature genes available: {len(sig_genes)} / {len(set(g for gs in LM22_markers.values() for g in gs))}')

sig_matrix = pd.DataFrame(0.0, index=sig_genes, columns=cell_types)
for cell_type, markers in LM22_markers.items():
    for gene in markers:
        if gene in sig_genes:
            sig_matrix.loc[gene, cell_type] = 1.0

# Expression matrix aligned to signature genes
expr_sig = lm22_expr_df.loc[sig_genes] if sig_genes else pd.DataFrame()

print('Signature matrix shape:', sig_matrix.shape)
print('Expression matrix shape:', expr_sig.shape)

# NNLS deconvolution per sample
A = sig_matrix.values  # genes x cell_types
fraction_rows = []

for sample in expr_sig.columns:
    b    = expr_sig[sample].values
    x, _ = nnls(A, b)
    # normalize to sum to 1
    total = x.sum()
    x_norm = x / total if total > 0 else x
    fraction_rows.append(x_norm)

fractions_df = pd.DataFrame(fraction_rows, index=expr_sig.columns, columns=cell_types)
fractions_df.index.name = 'sample'

print('\nImmune fractions shape:', fractions_df.shape)
print('Mean fractions per cell type:')
print(fractions_df.mean().sort_values(ascending=False).head(10))

Signature genes available: 40 / 80
Signature matrix shape: (40, 22)
Expression matrix shape: (40, 121)

Immune fractions shape: (121, 22)
Mean fractions per cell type:
T_follicular_helper     0.088882
NK_activated            0.086664
B_memory                0.079786
CD8_T_cells             0.078567
Mast_resting            0.065201
DC_resting              0.057916
CD4_memory_activated    0.048097
CD4_naive_T             0.047856
Monocytes               0.045895
CD4_memory_resting      0.044580
dtype: float64


In [75]:
# 21. compute exhaustion composite score (LAG3, HAVCR2/TIM3, TIGIT, PDCD1)

exhaust_genes_use = [g for g in ['LAG3','HAVCR2','TIGIT','PDCD1'] if g in lm22_expr_df.index]
print('Exhaustion genes found:', exhaust_genes_use)

if exhaust_genes_use:
    # z-score each gene across 121 samples, then take mean
    exhaust_z = lm22_expr_df.loc[exhaust_genes_use].apply(
        lambda row: (row - row.mean()) / (row.std() + 1e-8), axis=1
    )
    exhaustion_composite = exhaust_z.mean(axis=0)
    exhaustion_composite.name = 'exhaustion_composite'
    print('Exhaustion composite range:', round(exhaustion_composite.min(),3),
          'to', round(exhaustion_composite.max(),3))
else:
    exhaustion_composite = pd.Series(np.nan, index=fractions_df.index, name='exhaustion_composite')
    print('No exhaustion genes found; composite = NaN')

Exhaustion genes found: ['LAG3', 'HAVCR2']
Exhaustion composite range: -1.68 to 2.636


In [76]:
# 22. merge immune fractions and exhaustion with GIPS group labels

fractions_reset = fractions_df.reset_index()
fractions_reset['exhaustion_composite'] = exhaustion_composite.reindex(fractions_df.index).values

immune_deconv = fractions_reset.merge(gips_df[['sample','GIPS_group','GIPS_scaled']], on='sample', how='inner')

print('Merged shape:', immune_deconv.shape)
print('GIPS groups:', immune_deconv['GIPS_group'].value_counts().to_dict())

# Statistical tests: Kruskal-Wallis across all 3 GIPS groups
test_cols = cell_types + ['exhaustion_composite']
kw_results = []

for col in test_cols:
    if col not in immune_deconv.columns:
        continue
    groups = [immune_deconv[immune_deconv['GIPS_group']==g][col].dropna().values
              for g in group_order]
    groups = [g for g in groups if len(g) > 1]
    if len(groups) < 2:
        continue
    try:
        h, p = kruskal(*groups)
        # pairwise low vs high if p < 0.1
        lo = immune_deconv[immune_deconv['GIPS_group']=='Low'][col].dropna().values
        hi = immune_deconv[immune_deconv['GIPS_group']=='High'][col].dropna().values
        mwu_p = mannwhitneyu(lo, hi, alternative='two-sided').pvalue if len(lo)>1 and len(hi)>1 else 1.0
        kw_results.append({'cell_type': col, 'KW_H': round(h,3), 'KW_p': round(p,4),
                           'MWU_low_vs_high_p': round(mwu_p,4)})
    except:
        pass

kw_df = pd.DataFrame(kw_results).sort_values('KW_p')
print('\nTop immune associations with GIPS:')
print(kw_df.head(10))

Merged shape: (121, 26)
GIPS groups: {'Low': 52, 'High': 36, 'Moderate': 33}

Top immune associations with GIPS:
               cell_type    KW_H    KW_p  MWU_low_vs_high_p
8           NK_activated  11.684  0.0029             0.0013
1            CD4_naive_T  11.076  0.0039             0.0007
14        M1_macrophages  10.480  0.0053             0.0014
6          T_gamma_delta  10.236  0.0060             0.0052
3   CD4_memory_activated   9.149  0.0103             0.0056
4    T_follicular_helper   8.200  0.0166             0.0074
19        Mast_activated   7.807  0.0202             0.0042
11          Plasma_cells   7.535  0.0231             0.0165
0            CD8_T_cells   7.176  0.0277             0.0064
7             NK_resting   6.486  0.0391             0.0177


In [77]:
# 23. boxplots: CD8+ T, M1/M2, NK, Treg, exhaustion

plot_cell_types = ['CD8_T_cells','M1_macrophages','M2_macrophages',
                   'NK_activated','T_regulatory','exhaustion_composite']
plot_cell_types = [c for c in plot_cell_types if c in immune_deconv.columns]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cell_types):
    ax = axes[i]
    data = [immune_deconv[immune_deconv['GIPS_group']==g][col].dropna().values
            for g in group_order]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops=dict(color='black',linewidth=1.5))
    for patch, grp in zip(bp['boxes'], group_order):
        patch.set_facecolor(group_colors[grp])
        patch.set_alpha(0.7)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(group_order, fontsize=9)
    title = col.replace('_',' ')
    kw_row = kw_df[kw_df['cell_type']==col]
    p_str  = f'p={kw_row["KW_p"].values[0]}' if len(kw_row)>0 else ''
    ax.set_title(f'{title}\n{p_str}', fontsize=9)
    ax.set_ylabel('fraction', fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Immune cell fractions (CIBERSORT NNLS) by GIPS group', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb5_immune_deconv_boxplots.png'), dpi=150)
plt.show()
print('saved immune deconvolution boxplots')

saved immune deconvolution boxplots


In [78]:
# 24. save immune deconvolution results

immune_deconv.to_csv(os.path.join(tables_dir, 'immune_deconvolution_gips.csv'), index=False)
kw_df.to_csv(os.path.join(tables_dir, 'immune_deconv_stats.csv'), index=False)
print('Saved immune_deconvolution_gips.csv')
print('Saved immune_deconv_stats.csv')
print('Significant cell types (KW p<0.1):', kw_df[kw_df['KW_p']<0.1]['cell_type'].tolist())

Saved immune_deconvolution_gips.csv
Saved immune_deconv_stats.csv
Significant cell types (KW p<0.1): ['NK_activated', 'CD4_naive_T', 'M1_macrophages', 'T_gamma_delta', 'CD4_memory_activated', 'T_follicular_helper', 'Mast_activated', 'Plasma_cells', 'CD8_T_cells', 'NK_resting', 'DC_activated', 'B_naive']


In [79]:
# 25. Section 4: METABRIC identifier fix and completed survival analysis
# Issue in notebook 4: merge used 'Study ID' (same for all rows) instead of 'Sample ID'
# Fix: merge on 'Sample ID' column

metabric_clin  = pd.read_csv(os.path.join(data_dir, 'brca_metabric_clinical_data.tsv'), sep='\t')
metabric_tar   = os.path.join(data_dir, 'brca_metabric.tar.gz')
mb_extract_dir = os.path.join(base, 'results', 'metabric_extracted')

# Confirm sample ID format
print('Clinical columns (first 5):', list(metabric_clin.columns[:5]))
print('Sample ID examples:', metabric_clin['Sample ID'].head(5).tolist())

# Load expression matrix (already extracted)
mb_expr_path = os.path.join(mb_extract_dir, 'brca_metabric', 'data_mrna_illumina_microarray.txt')
mb_expr      = pd.read_csv(mb_expr_path, sep='\t', index_col=0)
if 'Entrez_Gene_Id' in mb_expr.columns:
    mb_expr = mb_expr.drop(columns=['Entrez_Gene_Id'])

print('METABRIC expression shape:', mb_expr.shape)
print('Expression column examples:', list(mb_expr.columns[:5]))
print('Match rate: Sample ID in expression columns:', 
      sum(1 for s in metabric_clin['Sample ID'] if s in mb_expr.columns), '/', len(metabric_clin))

Clinical columns (first 5): ['Study ID', 'Patient ID', 'Sample ID', 'Age at Diagnosis', 'Type of Breast Surgery']
Sample ID examples: ['MB-0000', 'MB-0002', 'MB-0005', 'MB-0006', 'MB-0008']
METABRIC expression shape: (20603, 1980)
Expression column examples: ['MB-0362', 'MB-0346', 'MB-0386', 'MB-0574', 'MB-0185']
Match rate: Sample ID in expression columns: 1980 / 2509


In [80]:
# 26. Expand METABRIC TNBC to include Basal + claudin-low
# Extension plan: 'retaining samples annotated as Basal or claudin-low'

pam50_col = 'Pam50 + Claudin-low subtype'
mb_tnbc_clin = metabric_clin[
    metabric_clin[pam50_col].isin(['Basal', 'claudin-low'])
].copy()

print('METABRIC TNBC-equivalent samples (Basal + claudin-low):', len(mb_tnbc_clin))
print('Breakdown:', mb_tnbc_clin[pam50_col].value_counts().to_dict())

mb_tnbc_ids  = mb_tnbc_clin['Sample ID'].tolist()
mb_tnbc_expr = [s for s in mb_tnbc_ids if s in mb_expr.columns]
print('TNBC samples with expression data:', len(mb_tnbc_expr))

# Load CN
mb_cn_path = os.path.join(mb_extract_dir, 'brca_metabric', 'data_cna.txt')
mb_cn      = pd.read_csv(mb_cn_path, sep='\t', index_col=0)
if 'Entrez_Gene_Id' in mb_cn.columns:
    mb_cn = mb_cn.drop(columns=['Entrez_Gene_Id'])
mb_tnbc_cn = [s for s in mb_tnbc_expr if s in mb_cn.columns]
print('TNBC samples with CN data:', len(mb_tnbc_cn))

METABRIC TNBC-equivalent samples (Basal + claudin-low): 427
Breakdown: {'claudin-low': 218, 'Basal': 209}
TNBC samples with expression data: 427
TNBC samples with CN data: 427


In [82]:
# 27. compute 2-modality GIPS for expanded METABRIC TNBC cohort

common_mb = mb_tnbc_cn  # samples with both expression and CN
panel_in_mb = [g for g in all_panel if g in mb_expr.index]

mb_expr_sub = mb_expr.loc[panel_in_mb, common_mb]
mb_cn_sub   = mb_cn.loc[panel_in_mb, common_mb]

print('METABRIC expression subset:', mb_expr_sub.shape)
print('METABRIC CN subset:', mb_cn_sub.shape)

# Expression z-score disruption
mb_z = mb_expr_sub.apply(lambda row: (row - row.mean())/(row.std()+1e-8), axis=1)
mb_expr_disr = (mb_z.abs() > 1.96).astype(float)

# CN disruption
mb_cn_disr = (mb_cn_sub != 0).astype(float)

# 2-modality gene score
mb_gene_score2 = (mb_expr_disr + mb_cn_disr) / 2.0

# GIPS
mb_gips2_raw    = mb_gene_score2.sum(axis=0)
mb_gips2_scaled = (mb_gips2_raw - mb_gips2_raw.min()) / (mb_gips2_raw.max() - mb_gips2_raw.min())
mb_t33 = mb_gips2_scaled.quantile(0.333)
mb_t66 = mb_gips2_scaled.quantile(0.667)

def mb_group(x):
    if x <= mb_t33: return 'Low'
    elif x <= mb_t66: return 'Moderate'
    return 'High'

mb_gips2_df = pd.DataFrame({
    'sample':      common_mb,
    'GIPS_scaled': mb_gips2_scaled.values,
    'GIPS_group':  mb_gips2_scaled.apply(mb_group).values,
})

print('METABRIC GIPS groups:', mb_gips2_df['GIPS_group'].value_counts().to_dict())

METABRIC expression subset: (24, 427)
METABRIC CN subset: (23, 427)
METABRIC GIPS groups: {'Low': 143, 'Moderate': 143, 'High': 141}


In [83]:
# 28. fix METABRIC survival merge: use 'Sample ID' column

os_time_col  = 'Overall Survival (Months)'
os_event_col = 'Overall Survival Status'
rfs_time_col = 'Relapse Free Status (Months)'
rfs_event_col = 'Relapse Free Status'

# Build survival table using Sample ID
mb_surv = metabric_clin[['Sample ID', os_time_col, os_event_col,
                           rfs_time_col, rfs_event_col]].copy()
mb_surv.columns = ['sample', 'os_time', 'os_event', 'rfs_time', 'rfs_event']

# Encode events
mb_surv['os_event_bin']  = mb_surv['os_event'].astype(str).str.contains('1:|DECEASED|died', case=False).astype(int)
mb_surv['rfs_event_bin'] = mb_surv['rfs_event'].astype(str).str.contains('1:|Recurred|relapsed', case=False).astype(int)
mb_surv['os_time']  = pd.to_numeric(mb_surv['os_time'], errors='coerce')
mb_surv['rfs_time'] = pd.to_numeric(mb_surv['rfs_time'], errors='coerce')

mb_surv_gips = mb_gips2_df.merge(mb_surv, on='sample', how='inner')

print('METABRIC survival+GIPS shape (FIXED):', mb_surv_gips.shape)
print('OS events:', int(mb_surv_gips['os_event_bin'].sum()))
print('RFS events:', int(mb_surv_gips['rfs_event_bin'].sum()))
print('GIPS groups:', mb_surv_gips['GIPS_group'].value_counts().to_dict())

METABRIC survival+GIPS shape (FIXED): (427, 9)
OS events: 209
RFS events: 427
GIPS groups: {'Low': 143, 'Moderate': 143, 'High': 141}


In [84]:
# 29. METABRIC Kaplan-Meier survival

from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test, logrank_test

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, time_col, event_col, label in [
    (axes[0], 'os_time',  'os_event_bin',  'OS'),
    (axes[1], 'rfs_time', 'rfs_event_bin', 'RFS'),
]:
    valid = mb_surv_gips.dropna(subset=[time_col, event_col])
    kmf = KaplanMeierFitter()
    for grp in group_order:
        sub = valid[valid['GIPS_group']==grp]
        if len(sub) < 3:
            continue
        kmf.fit(sub[time_col], sub[event_col], label=f'{grp} (n={len(sub)})')
        kmf.plot_survival_function(ax=ax, ci_show=True, color=group_colors[grp])
    res = multivariate_logrank_test(valid[time_col], valid['GIPS_group'], valid[event_col])
    # pairwise low vs high
    lo = valid[valid['GIPS_group']=='Low']
    hi = valid[valid['GIPS_group']=='High']
    if len(lo)>3 and len(hi)>3:
        pw = logrank_test(lo[time_col], hi[time_col], lo[event_col], hi[event_col])
        pw_str = f', Low vs High p={round(pw.p_value,4)}'
    else:
        pw_str = ''
    ax.set_title(f'METABRIC {label} by GIPS (p={round(res.p_value,4)}{pw_str})', fontsize=9)
    ax.set_xlabel('months')
    ax.set_ylabel('survival probability')
    ax.legend(fontsize=8)

plt.suptitle(f'METABRIC TNBC-equivalent (Basal+claudin-low, n={len(mb_surv_gips)})', fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb5_metabric_km.png'), dpi=150)
plt.show()
print('saved METABRIC KM curves')

saved METABRIC KM curves


In [85]:
# 30. METABRIC Cox regression

from lifelines import CoxPHFitter

cox_results = []
for time_col, event_col, label in [
    ('os_time', 'os_event_bin', 'OS'),
    ('rfs_time', 'rfs_event_bin', 'RFS'),
]:
    valid = mb_surv_gips[['GIPS_scaled', time_col, event_col]].dropna()
    valid.columns = ['GIPS', 'duration', 'event']
    try:
        cph = CoxPHFitter()
        cph.fit(valid, duration_col='duration', event_col='event')
        s = cph.summary
        hr  = round(np.exp(s['coef'].values[0]), 3)
        ci_lo = round(np.exp(s['coef lower 95%'].values[0]), 3)
        ci_hi = round(np.exp(s['coef upper 95%'].values[0]), 3)
        p   = round(s['p'].values[0], 4)
        print(f'{label}: HR={hr} (95% CI: {ci_lo}-{ci_hi}), p={p}')
        cox_results.append({'endpoint': label, 'HR': hr, 'CI_low': ci_lo, 'CI_high': ci_hi, 'p': p})
    except Exception as ex:
        print(f'{label}: Cox error: {ex}')

cox_mb_df = pd.DataFrame(cox_results)
cox_mb_df.to_csv(os.path.join(tables_dir, 'metabric_cox_results.csv'), index=False)

OS: HR=1.381 (95% CI: 0.811-2.353), p=0.2342
RFS: HR=1.028 (95% CI: 0.698-1.515), p=0.8881


In [86]:
# 31. cross-cohort Spearman correlation (corrected sample mapping)

# Gene-level mean disruption in TCGA-BRCA TNBC
tcga_gene_mean = gene_score.mean(axis=1)
tcga_gene_mean = tcga_gene_mean[~tcga_gene_mean.index.duplicated(keep='first')]

# Gene-level mean disruption in METABRIC (corrected, Basal+claudin-low)
mb_gene_mean = mb_gene_score2.mean(axis=1)
mb_gene_mean = mb_gene_mean[~mb_gene_mean.index.duplicated(keep='first')]

common_genes = sorted(set(tcga_gene_mean.index) & set(mb_gene_mean.index))
print('Common genes for cross-cohort correlation:', len(common_genes))

tcga_vals = tcga_gene_mean[common_genes].values
mb_vals   = mb_gene_mean[common_genes].values
r, p      = spearmanr(tcga_vals, mb_vals)

print(f'Cross-cohort Spearman r = {round(r,3)}, p = {round(p,4)}')

# scatter plot
fig, ax = plt.subplots(figsize=(7, 6))
colors  = ['#d94f3d' if g in hr_genes else '#f0a500' if g in cohesin_genes else '#4878cf' for g in common_genes]
ax.scatter(tcga_vals, mb_vals, c=colors, s=60, alpha=0.8, edgecolors='white', linewidths=0.5)
for i, g in enumerate(common_genes):
    ax.annotate(g, (tcga_vals[i], mb_vals[i]), fontsize=7, alpha=0.7)
ax.set_xlabel('TCGA-BRCA TNBC mean disruption score')
ax.set_ylabel('METABRIC (Basal+claudin-low) mean disruption score')
ax.set_title(f'Cross-cohort gene disruption concordance\nSpearman r={round(r,3)}, p={round(p,4)}')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#d94f3d', label='HR genes'),
                   Patch(facecolor='#f0a500', label='Cohesin genes'),
                   Patch(facecolor='#4878cf', label='Meiosis genes')]
ax.legend(handles=legend_elements, fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb5_cross_cohort_spearman.png'), dpi=150)
plt.show()
print(f'Cross-cohort concordance: r={round(r,3)}, p={round(p,4)}')

Common genes for cross-cohort correlation: 23
Cross-cohort Spearman r = 0.41, p = 0.0521
Cross-cohort concordance: r=0.41, p=0.0521


In [87]:
# 32. save METABRIC results and notebook 5 summary

mb_surv_gips.to_csv(os.path.join(tables_dir, 'metabric_gips_survival.csv'), index=False)

os_res  = multivariate_logrank_test(mb_surv_gips.dropna(subset=['os_time'])['os_time'],
                                     mb_surv_gips.dropna(subset=['os_time'])['GIPS_group'],
                                     mb_surv_gips.dropna(subset=['os_time'])['os_event_bin'])
rfs_res = multivariate_logrank_test(mb_surv_gips.dropna(subset=['rfs_time'])['rfs_time'],
                                     mb_surv_gips.dropna(subset=['rfs_time'])['GIPS_group'],
                                     mb_surv_gips.dropna(subset=['rfs_time'])['rfs_event_bin'])

summary5 = {
    'TNBC classification concordance pct':      round(concordance*100, 1),
    'Methylation probes found':                 found_count,
    'Methylation genes with data':              int((meth_disr[common_samps_meth].sum(axis=1)>0).sum()) if common_samps_meth else 0,
    'GIPS4 groups':                             str(gips4_df['GIPS4_group'].value_counts().to_dict()),
    'LM22 sig genes available':                 len(sig_genes),
    'Significant immune KW p<0.1':              int((kw_df['KW_p']<0.1).sum()),
    'METABRIC TNBC samples (Basal+claudin-low)': len(mb_gips2_df),
    'METABRIC OS log-rank p':                   round(os_res.p_value, 4),
    'METABRIC RFS log-rank p':                  round(rfs_res.p_value, 4),
    'Cross-cohort Spearman r':                  round(r, 3),
    'Cross-cohort Spearman p':                  round(p, 4),
}

for k, v in summary5.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(summary5, orient='index', columns=['value']).to_csv(
    os.path.join(tables_dir, 'nb5_summary.csv'))
print('notebook 5 complete')

TNBC classification concordance pct: 88.4
Methylation probes found: 122
Methylation genes with data: 17
GIPS4 groups: {'Low': 46, 'Moderate': 39, 'High': 36}
LM22 sig genes available: 40
Significant immune KW p<0.1: 12
METABRIC TNBC samples (Basal+claudin-low): 427
METABRIC OS log-rank p: 0.3114
METABRIC RFS log-rank p: 0.7812
Cross-cohort Spearman r: 0.41
Cross-cohort Spearman p: 0.0521
notebook 5 complete
